# Sagewai — Example 44 — Free CUDA via Colab + Drive-sync

**Companion notebook for `44_colab_free_cuda.py`.**

This notebook fine-tunes a 4-bit Llama-3.2-3B with a LoRA on the email-triage JSONL the orchestrator uploaded to your Drive, evaluates the result on a held-out set, and writes the LoRA back to the same Drive folder so the orchestrator can download it.

**To run:**

1. Make sure the runtime is set to **T4 GPU** (`Runtime → Change runtime type → T4 GPU`).
2. Click **Runtime → Run all**.
3. Walk away. The notebook installs Unsloth, finds the latest `SagewaiTraining/run-*/` folder in your Drive, trains, evaluates, and uploads the LoRA. ~6-10 minutes end-to-end on T4.

**Cost:** $0.00 — Colab's free tier.

## 1. Install Unsloth + dependencies

Colab's pre-installed PyTorch + CUDA stack is the right baseline; we only need Unsloth + TRL + datasets on top. The `unsloth` install script picks the right wheel for the runtime.

In [ ]:
%pip install --quiet --upgrade pip
%pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install --quiet --no-deps "trl<0.9.0" peft accelerate bitsandbytes
%pip install --quiet datasets

## 2. Mount Google Drive

Mounts your personal Drive at `/content/drive`. You'll be asked to grant Colab access — it's the same Google account you're already signed in with.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Locate the latest run folder

The orchestrator (`44_colab_free_cuda.py --live`) creates `SagewaiTraining/run-<timestamp>/` for each run and uploads the JSONL into it. We pick the most recent one — re-runs of this notebook against a fresh orchestrator run pick up the new folder automatically.

In [ ]:
import os, json, glob, tarfile, time
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/SagewaiTraining'
candidates = sorted(glob.glob(os.path.join(DRIVE_ROOT, 'run-*')))
assert candidates, (
    f'No run folders found under {DRIVE_ROOT}. '
    'Run `python 44_colab_free_cuda.py --live` from your laptop first.'
)
RUN_DIR = candidates[-1]
OUTPUT_DIR = os.path.join(RUN_DIR, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Using run folder:', RUN_DIR)
print('Files present   :', os.listdir(RUN_DIR))

## 4. Load training + held-out eval data

In [ ]:
from datasets import Dataset

with open(os.path.join(RUN_DIR, 'email_triage.jsonl')) as fh:
    train_samples = [json.loads(line) for line in fh if line.strip()]
with open(os.path.join(RUN_DIR, 'email_triage_eval.jsonl')) as fh:
    eval_samples = [json.loads(line) for line in fh if line.strip()]

def to_alpaca(ex):
    return {
        'text': (
            f"### Instruction:\n{ex['instruction']}\n\n"
            f"### Input:\n{ex['input']}\n\n"
            f"### Response:\n{ex['output']}"
        ),
    }

train_dataset = Dataset.from_list([to_alpaca(s) for s in train_samples])
print('train samples :', len(train_dataset))
print('eval samples  :', len(eval_samples))

## 5. Load the 4-bit base model and attach a LoRA

Hyper-params mirror Examples 38 + 47 so the LoRA is identical across the inference spectrum: `r=16`, `α=32`, dropout `0.05`, target attention projections.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-3B-Instruct-bnb-4bit',
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

## 6. Fine-tune (1 epoch, ~5 minutes on T4)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        num_train_epochs=1,
        learning_rate=2e-4,
        output_dir='/content/output',
        logging_steps=1,
        save_strategy='epoch',
        fp16=True,
    ),
)
train_started = time.time()
trainer.train()
train_seconds = time.time() - train_started
print(f'Training wall: {train_seconds:.1f}s')

## 7. Evaluate on the held-out set

We classify each held-out email by greedy-decoding from the fine-tuned model and parsing the JSON response. The notebook reports accuracy as `correct/total`; the orchestrator reads this from `output/eval_result.json` and gates the success message on the ≥80% target from the issue's acceptance criterion.

In [ ]:
import re, torch
FastLanguageModel.for_inference(model)

def classify(input_text: str) -> str:
    prompt = (
        '### Instruction:\nClassify the urgency of this customer-support email.\n\n'
        f'### Input:\n{input_text}\n\n### Response:\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=64,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    m = re.search(r'"urgency"\s*:\s*"(\w+)"', text)
    if m:
        return m.group(1).lower()
    # Fallback: pick the first urgency-shaped word; this is the loose path.
    for token in ('high', 'medium', 'low'):
        if token in text.lower():
            return token
    return 'unknown'

results = []
correct = 0
for ex in eval_samples:
    predicted = classify(ex['input'])
    ok = (predicted == ex['expected'])
    correct += int(ok)
    results.append({'expected': ex['expected'], 'predicted': predicted, 'correct': ok})

accuracy = correct / len(eval_samples)
print(f'Eval: {correct}/{len(eval_samples)} correct ({accuracy:.0%})')
for r in results:
    mark = '✓' if r['correct'] else '✗'
    print(f'  {mark}  expected={r["expected"]:<6}  predicted={r["predicted"]}')

## 8. Save + tar the LoRA, write eval result, upload back to Drive

The orchestrator polls `output/lora.tar.gz` in this folder; landing the tarball is what completes the pipeline.

In [ ]:
lora_dir = '/content/output/lora'
os.makedirs(lora_dir, exist_ok=True)
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)

tar_path = os.path.join(OUTPUT_DIR, 'lora.tar.gz')
with tarfile.open(tar_path, 'w:gz') as tf:
    tf.add(lora_dir, arcname='lora')

eval_blob = {
    'correct': correct,
    'total': len(eval_samples),
    'accuracy': accuracy,
    'train_seconds': train_seconds,
    'results': results,
}
with open(os.path.join(OUTPUT_DIR, 'eval_result.json'), 'w') as fh:
    json.dump(eval_blob, fh, indent=2)

print('LoRA written to:', tar_path)
print('Eval written to:', os.path.join(OUTPUT_DIR, 'eval_result.json'))
print()
print('Switch back to the orchestrator terminal — it will detect the LoRA')
print('within ~30s and download it. Total Colab spend: $0.00.')